<a href="https://colab.research.google.com/github/2303a51514/reinforcement-learning/blob/main/lab-10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Practical 10: Bellman Optimality and Value Iteration
# Task 1: Inspect the MDP

# Define states
states = ['S0', 'S1', 'G']
non_terminal_states = ['S0', 'S1']

# Define actions
actions = ['Up', 'Down', 'Left', 'Right']

# Define discount factor and rewards
gamma = 0.9
normal_reward = -1
wall_reward = -1
goal_reward = 10

# Define transition mapping
transitions = {
    'S0': {
        'Up': ('S0', wall_reward),
        'Down': ('S1', normal_reward),
        'Left': ('S0', wall_reward),
        'Right': ('S0', wall_reward)
    },
    'S1': {
        'Up': ('S0', normal_reward),
        'Down': ('S1', wall_reward),
        'Left': ('S1', wall_reward),
        'Right': ('G', goal_reward)
    }
}

# Display MDP information
print("States:", states)
print("Actions:", actions)
print("Discount Factor:", gamma)
print("Normal Movement Reward:", normal_reward)
print("Wall Reward:", wall_reward)
print("Goal Reward:", goal_reward)
print("\nTransition Mapping:")

for state in non_terminal_states:
    for action in actions:
        next_state, reward = transitions[state][action]
        print(f"{state} --{action}--> {next_state}, Reward = {reward}")

States: ['S0', 'S1', 'G']
Actions: ['Up', 'Down', 'Left', 'Right']
Discount Factor: 0.9
Normal Movement Reward: -1
Wall Reward: -1
Goal Reward: 10

Transition Mapping:
S0 --Up--> S0, Reward = -1
S0 --Down--> S1, Reward = -1
S0 --Left--> S0, Reward = -1
S0 --Right--> S0, Reward = -1
S1 --Up--> S0, Reward = -1
S1 --Down--> S1, Reward = -1
S1 --Left--> S1, Reward = -1
S1 --Right--> G, Reward = 10


In [3]:
# Task 2: Verify all transitions

all_defined = True

for state in non_terminal_states:
    for action in actions:
        if action in transitions[state]:
            next_state, reward = transitions[state][action]

            print("Current State:", state)
            print("Action:", action)
            print("Next State:", next_state)
            print("Immediate Reward:", reward)
            print("-------------------------")
        else:
            all_defined = False
            print("Missing transition:", state, action)

print("All transitions defined:", all_defined)

Current State: S0
Action: Up
Next State: S0
Immediate Reward: -1
-------------------------
Current State: S0
Action: Down
Next State: S1
Immediate Reward: -1
-------------------------
Current State: S0
Action: Left
Next State: S0
Immediate Reward: -1
-------------------------
Current State: S0
Action: Right
Next State: S0
Immediate Reward: -1
-------------------------
Current State: S1
Action: Up
Next State: S0
Immediate Reward: -1
-------------------------
Current State: S1
Action: Down
Next State: S1
Immediate Reward: -1
-------------------------
Current State: S1
Action: Left
Next State: S1
Immediate Reward: -1
-------------------------
Current State: S1
Action: Right
Next State: G
Immediate Reward: 10
-------------------------
All transitions defined: True


In [4]:
# Task 3: Initialize state values

V = {
    'S0': 0.0,
    'S1': 0.0,
    'G': 0.0
}

print("Initial State Values:")

for state, value in V.items():
    print(f"V({state}) = {value}")

Initial State Values:
V(S0) = 0.0
V(S1) = 0.0
V(G) = 0.0


In [5]:
# Task 4: Synchronous Value Iteration

theta = 0.00001
iteration = 0

while True:
    # Copy previous state values
    new_V = V.copy()

    # Calculate new values using previous iteration values
    for state in non_terminal_states:
        q_values = []

        for action in actions:
            next_state, reward = transitions[state][action]
            q = reward + gamma * V[next_state]
            q_values.append(q)

        new_V[state] = max(q_values)

    # Goal state is terminal
    new_V['G'] = 0.0

    # Calculate maximum change
    delta = max(
        abs(new_V[state] - V[state])
        for state in states
    )

    iteration += 1
    V = new_V

    print(f"Iteration {iteration}: {V}, Delta = {delta:.6f}")

    # Check convergence
    if delta < theta:
        break

print("\nFinal State Values:", V)

Iteration 1: {'S0': -1.0, 'S1': 10.0, 'G': 0.0}, Delta = 10.000000
Iteration 2: {'S0': 8.0, 'S1': 10.0, 'G': 0.0}, Delta = 9.000000
Iteration 3: {'S0': 8.0, 'S1': 10.0, 'G': 0.0}, Delta = 0.000000

Final State Values: {'S0': 8.0, 'S1': 10.0, 'G': 0.0}


In [6]:
# Task 5: Record convergence

V_history = {s: 0.0 for s in states}
theta = 0.00001
iteration = 0

print("Iteration | V(S0) | V(S1) | V(G) | Delta")

while True:
    new_V = V_history.copy()

    for state in non_terminal_states:
        q_values = []

        for action in actions:
            next_state, reward = transitions[state][action]
            q = reward + gamma * V_history[next_state]
            q_values.append(q)

        new_V[state] = max(q_values)

    new_V['G'] = 0.0

    delta = max(
        abs(new_V[s] - V_history[s])
        for s in states
    )

    iteration += 1

    print(
        f"{iteration:9d} | {new_V['S0']:5.2f} | "
        f"{new_V['S1']:5.2f} | {new_V['G']:5.2f} | {delta:.5f}"
    )

    V_history = new_V

    if delta < theta:
        break

print("\nConverged State Values:", V_history)
print("Total Iterations:", iteration)

Iteration | V(S0) | V(S1) | V(G) | Delta
        1 | -1.00 | 10.00 |  0.00 | 10.00000
        2 |  8.00 | 10.00 |  0.00 | 9.00000
        3 |  8.00 | 10.00 |  0.00 | 0.00000

Converged State Values: {'S0': 8.0, 'S1': 10.0, 'G': 0.0}
Total Iterations: 3


In [7]:
# Task 6: Calculate action values

Q = {}

for state in non_terminal_states:
    Q[state] = {}

    for action in actions:
        next_state, reward = transitions[state][action]

        q_value = reward + gamma * V_history[next_state]
        Q[state][action] = q_value

print("Optimal Action Values Q*(s,a):")

for state in non_terminal_states:
    print(f"\nState: {state}")

    for action in actions:
        print(f"Q*({state}, {action}) = {Q[state][action]:.2f}")

Optimal Action Values Q*(s,a):

State: S0
Q*(S0, Up) = 6.20
Q*(S0, Down) = 8.00
Q*(S0, Left) = 6.20
Q*(S0, Right) = 6.20

State: S1
Q*(S1, Up) = 6.20
Q*(S1, Down) = 8.00
Q*(S1, Left) = 8.00
Q*(S1, Right) = 10.00


In [8]:
# Task 7: Extract the optimal policy

policy = {}

for state in non_terminal_states:
    max_q = max(Q[state].values())

    best_actions = [
        action for action in actions
        if Q[state][action] == max_q
    ]

    policy[state] = best_actions

print("Optimal Policy:")

for state in non_terminal_states:
    print(f"{state}: {policy[state]}")

Optimal Policy:
S0: ['Down']
S1: ['Right']


In [9]:
# Task 8: Verify Bellman Optimality

print("Bellman Optimality Verification:")

for state in non_terminal_states:
    state_value = V_history[state]
    max_q_value = max(Q[state].values())

    verified = abs(state_value - max_q_value) < 1e-6

    print(f"\nState: {state}")
    print(f"V*({state}) = {state_value:.2f}")
    print(f"Maximum Q-value = {max_q_value:.2f}")
    print("Verified:", verified)

Bellman Optimality Verification:

State: S0
V*(S0) = 8.00
Maximum Q-value = 8.00
Verified: True

State: S1
V*(S1) = 10.00
Maximum Q-value = 10.00
Verified: True


In [10]:
# Task 9: Final Results and Explanation

print("FINAL RESULTS")
print("-------------------------")

print("\nOptimal State Values:")
for state in states:
    print(f"V*({state}) = {V_history[state]:.2f}")

print("\nOptimal Policy:")
for state in non_terminal_states:
    print(f"{state} -> {policy[state]}")

print("\nExplanation:")
print("1. Down is optimal at S0.")
print("2. Right is optimal at S1 because it reaches the goal.")
print("3. The goal reward increases the value of reaching G.")
print("4. The discount factor gamma = 0.9 reduces future value contributions.")
print("5. Bellman Optimality is verified for both non-terminal states.")

FINAL RESULTS
-------------------------

Optimal State Values:
V*(S0) = 8.00
V*(S1) = 10.00
V*(G) = 0.00

Optimal Policy:
S0 -> ['Down']
S1 -> ['Right']

Explanation:
1. Down is optimal at S0.
2. Right is optimal at S1 because it reaches the goal.
3. The goal reward increases the value of reaching G.
4. The discount factor gamma = 0.9 reduces future value contributions.
5. Bellman Optimality is verified for both non-terminal states.
